# SolarSDE Notebook 2 — Neural SDE + Score Decoder + Main Evaluation

**Runtime:** ~1-2 hours on Colab T4 / Kaggle P100

**Prerequisite:** Notebook 1 must have been run successfully. This notebook reads latents from Google Drive (Colab) or /kaggle/working (Kaggle).

**This notebook:**
1. Loads latents, CTI, GHI, covariates from Notebook 1
2. Trains the Latent Neural SDE (100 epochs, SDE Matching)
3. Trains the Conditional Score-Matching Decoder (40 epochs)
4. Runs full probabilistic forecasting pipeline at 5 horizons (1, 5, 10, 20, 30 min)
5. Saves SolarSDE results for the main comparison table


In [ ]:
# ==== Install dependencies ====
import subprocess, sys
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

pip_install("pvlib", "properscoring", "pyarrow", "tqdm")
print("Dependencies installed.")


In [ ]:
# ==== Environment Detection & Setup ====
import os, sys, time, json, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")

print(f"Environment: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print(f"Drive mount issue: {e}")
    PERSIST_DIR = Path("/content/drive/MyDrive/solarsde_outputs")
    WORK_DIR = Path("/content/solarsde")
elif IN_KAGGLE:
    PERSIST_DIR = Path("/kaggle/working/solarsde_outputs")
    WORK_DIR = Path("/kaggle/working/solarsde")
else:
    PERSIST_DIR = Path.cwd() / "solarsde_outputs"
    WORK_DIR = Path.cwd() / "solarsde_work"

PERSIST_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = WORK_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = PERSIST_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = PERSIST_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LATENT_DIR = PERSIST_DIR / "latents"
LATENT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Persistent storage: {PERSIST_DIR}")
print(f"Working directory:  {WORK_DIR}")


In [ ]:
# ==== GPU Check ====
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device:          {torch.cuda.get_device_name(0)}")
    print(f"Memory:          {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
else:
    DEVICE = torch.device("cpu")
    print("WARNING: Running on CPU — will be slow. Change Runtime > Change runtime type > GPU")
print(f"Using device: {DEVICE}")


## Fast-start — pull Notebook 1 outputs from GitHub (skips VAE retraining)

In [ ]:
# ==== Fast-start: fetch Notebook 1 outputs from GitHub if persistent storage is empty ====
# This lets Notebooks 2-5 run standalone without having re-executed Notebook 1.
GITHUB_REPO = "https://github.com/keshavkrishnan08/SDE"
GITHUB_RAW = "https://raw.githubusercontent.com/keshavkrishnan08/SDE/main"

need_nb1_outputs = not (CHECKPOINT_DIR / "vae_best.pt").exists() or not any(LATENT_DIR.glob("train_*.npy"))
if need_nb1_outputs:
    print("Notebook 1 outputs not found in persistent storage — pulling from GitHub ...")
    import requests
    files_to_pull = [
        ("checkpoints/vae_best.pt",          CHECKPOINT_DIR / "vae_best.pt"),
        ("checkpoints/vae_final.pt",         CHECKPOINT_DIR / "vae_final.pt"),
        ("results/vae_training_history.csv", RESULTS_DIR / "vae_training_history.csv"),
        ("splits/train.parquet",             PERSIST_DIR / "splits" / "train.parquet"),
        ("splits/val.parquet",               PERSIST_DIR / "splits" / "val.parquet"),
        ("splits/test.parquet",              PERSIST_DIR / "splits" / "test.parquet"),
    ]
    for split in ["train", "val", "test"]:
        for k in ["latents", "cti", "ghi", "covariates", "is_ramp"]:
            files_to_pull.append((f"latents/{split}_{k}.npy", LATENT_DIR / f"{split}_{k}.npy"))

    for rel, dest in files_to_pull:
        url = f"{GITHUB_RAW}/colab_outputs/{rel}"
        if dest.exists() and dest.stat().st_size > 0:
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        try:
            r = requests.get(url, timeout=120)
            if r.status_code == 200 and len(r.content) > 100:
                dest.write_bytes(r.content)
                print(f"  OK  {rel}  ({len(r.content)/1e6:.2f} MB)")
            else:
                print(f"  SKIP {rel}  (status {r.status_code})")
        except Exception as e:
            print(f"  FAIL {rel}: {e}")
    print("Fast-start pull complete.")
else:
    print("Notebook 1 outputs already present in persistent storage.")


## 1. Load latents from Notebook 1

In [ ]:
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import time

def load_split(split):
    return {
        "Z":    np.load(LATENT_DIR / f"{split}_latents.npy"),
        "cti":  np.load(LATENT_DIR / f"{split}_cti.npy"),
        "ghi":  np.load(LATENT_DIR / f"{split}_ghi.npy"),
        "cov":  np.load(LATENT_DIR / f"{split}_covariates.npy"),
        "ramp": np.load(LATENT_DIR / f"{split}_is_ramp.npy"),
    }

data = {s: load_split(s) for s in ["train", "val", "test"]}
for s, d in data.items():
    print(f"  {s}: Z={d['Z'].shape}, CTI range=[{d['cti'].min():.4f},{d['cti'].max():.4f}], "
          f"GHI=[{d['ghi'].min():.1f},{d['ghi'].max():.1f}], cov={d['cov'].shape}, ramp={int(d['ramp'].sum())}")

Z_DIM = data["train"]["Z"].shape[1]
C_DIM = max(1, data["train"]["cov"].shape[1])
print(f"\nLatent dim: {Z_DIM}, covariate dim: {C_DIM}")


## 2. Neural SDE model

In [ ]:
# ==== Neural SDE model (v2: diffusion floor + regularization) ====
#
# Key fixes from v1:
#  1. Diffusion FLOOR: forces sigma >= SIGMA_FLOOR to prevent SDE collapse on
#     clear-sky days (where one-step residual is near zero, old SDE learned sigma=0).
#  2. Log-diffusion loss: matches log(sigma^2) to log(residual) which is better
#     conditioned than MSE on squared sigma.
#  3. CTI-modulated floor: even the minimum diffusion scales with CTI so clear-sky
#     uncertainty stays narrow, cloudy-sky uncertainty grows.
#  4. Drift target: predict direct z_next (not rate) to avoid tiny-rate collapse.

SIGMA_FLOOR_BASE = 0.01     # per-dim minimum diffusion; covers model noise

class ResBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d), nn.SiLU(inplace=True), nn.Linear(d, d))
    def forward(self, x): return x + self.net(x)

class DriftNet(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, h=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim + 1 + c_dim, h), nn.SiLU(inplace=True),
            nn.Linear(h, h), nn.SiLU(inplace=True),
            ResBlock(h), ResBlock(h),
            nn.Linear(h, z_dim),
        )
    def forward(self, z, t, c):
        return self.net(torch.cat([z, t, c], dim=-1))

class CTIDiffNet(nn.Module):
    """CTI-gated diffusion with floor:
         sigma = SIGMA_FLOOR_BASE * (1 + 10*CTI) + Softplus(MLP(z) * Softplus(MLP(CTI)))
    Guarantees sigma >= SIGMA_FLOOR_BASE everywhere, and floor grows with CTI.
    """
    def __init__(self, z_dim=64, h=64, sigma_floor=SIGMA_FLOOR_BASE):
        super().__init__()
        self.sigma_floor = sigma_floor
        self.cti_gate = nn.Sequential(nn.Linear(1, h), nn.Softplus())
        self.state = nn.Sequential(nn.Linear(z_dim, h), nn.SiLU(inplace=True))
        self.out = nn.Sequential(nn.Linear(h, z_dim), nn.Softplus())
    def forward(self, z, cti):
        base_floor = self.sigma_floor * (1.0 + 10.0 * cti)   # CTI-scaled floor
        learned = self.out(self.state(z) * self.cti_gate(cti))
        return base_floor + learned

class LatentNeuralSDE(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, drift_h=256, diff_h=64, lambda_sigma=1.0):
        super().__init__()
        self.z_dim = z_dim
        self.lambda_sigma = lambda_sigma
        self.drift = DriftNet(z_dim, c_dim, drift_h)
        self.diffusion = CTIDiffNet(z_dim, diff_h)
    def forward(self, z, t, c, cti):
        return self.drift(z, t, c), self.diffusion(z, cti)
    def sde_matching_loss(self, z, zn, t, c, cti, dt=1.0):
        mu = self.drift(z, t, c)
        sigma = self.diffusion(z, cti)
        dz = (zn - z) / dt
        drift_l = F.mse_loss(mu, dz)
        # Log-space diffusion matching: avoids scale issues when residuals are tiny
        resid = (zn - z - mu * dt).pow(2) / dt + 1e-8
        log_diff_l = F.mse_loss(torch.log(sigma.pow(2) + 1e-8), torch.log(resid))
        return {"loss": drift_l + self.lambda_sigma * log_diff_l,
                "drift": drift_l, "diffusion": log_diff_l}


## 3. Latent sequence dataset

In [ ]:
class LatentSeqDataset(Dataset):
    def __init__(self, data):
        self.Z = data["Z"]; self.cti = data["cti"]; self.ghi = data["ghi"]; self.cov = data["cov"]
    def __len__(self): return max(0, len(self.Z) - 1)
    def __getitem__(self, i):
        return {
            "z_t":   torch.from_numpy(self.Z[i]).float(),
            "z_next": torch.from_numpy(self.Z[i+1]).float(),
            "cti":   torch.tensor(float(self.cti[i])),
            "ghi":   torch.tensor(float(self.ghi[i])),
            "cov":   torch.from_numpy(self.cov[i]).float() if self.cov.shape[1] > 0 else torch.zeros(C_DIM),
        }


## 4. Train Neural SDE (SDE Matching)

In [ ]:
torch.manual_seed(42); np.random.seed(42)
train_ds = LatentSeqDataset(data["train"])
val_ds   = LatentSeqDataset(data["val"])
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False)

sde = LatentNeuralSDE(z_dim=Z_DIM, c_dim=C_DIM, drift_h=256, diff_h=64, lambda_sigma=1.0).to(DEVICE)
opt = torch.optim.Adam(sde.parameters(), lr=1e-4)
EPOCHS_SDE = 150
print(f"SDE params: {sum(p.numel() for p in sde.parameters()):,}")

best_val = float("inf"); t0 = time.time(); hist = []
for ep in range(1, EPOCHS_SDE + 1):
    sde.train(); tl = td = ts = 0; n = 0
    for b in train_loader:
        z = b["z_t"].to(DEVICE); zn = b["z_next"].to(DEVICE)
        cti = b["cti"].to(DEVICE).unsqueeze(-1); c = b["cov"].to(DEVICE)
        t = torch.zeros(z.shape[0], 1, device=DEVICE)
        losses = sde.sde_matching_loss(z, zn, t, c, cti)
        opt.zero_grad(); losses["loss"].backward()
        torch.nn.utils.clip_grad_norm_(sde.parameters(), 1.0)
        opt.step()
        tl += losses["loss"].item(); td += losses["drift"].item(); ts += losses["diffusion"].item(); n += 1
    tl/=n; td/=n; ts/=n
    sde.eval(); vl = vn = 0
    with torch.no_grad():
        for b in val_loader:
            z = b["z_t"].to(DEVICE); zn = b["z_next"].to(DEVICE)
            cti = b["cti"].to(DEVICE).unsqueeze(-1); c = b["cov"].to(DEVICE)
            t = torch.zeros(z.shape[0], 1, device=DEVICE)
            vl += sde.sde_matching_loss(z, zn, t, c, cti)["loss"].item(); vn += 1
    vl /= max(vn, 1)
    hist.append({"epoch": ep, "train_loss": tl, "drift": td, "diffusion": ts, "val_loss": vl})
    if ep % 10 == 0 or ep == 1:
        print(f"Epoch {ep:3d}/{EPOCHS_SDE} | train={tl:.5f} (drift={td:.5f}, diff={ts:.5f}) | val={vl:.5f} | {(time.time()-t0)/60:.1f} min")
    if vl < best_val:
        best_val = vl
        torch.save(sde.state_dict(), CHECKPOINT_DIR / "sde_best.pt")

torch.save(sde.state_dict(), CHECKPOINT_DIR / "sde_final.pt")
pd.DataFrame(hist).to_csv(RESULTS_DIR / "sde_training_history.csv", index=False)
print(f"SDE training complete. Best val: {best_val:.5f}. Time: {(time.time()-t0)/60:.1f} min")


## 5. Score-matching decoder

In [ ]:
# ==== Conditional Score-Matching Irradiance Decoder (CSMID v2) ====
#
# BIG CHANGE FROM V1: Now predicts CLEAR-SKY INDEX k_t = GHI / GHI_clearsky
# instead of raw GHI. This:
#   1. Removes the ~600 W/m² solar-elevation trend that persistence trivially
#      captures, letting the model focus on the cloud-driven residual.
#   2. Stabilizes targets: k_t is in [0, 1.5], independent of time of day.
#   3. At inference, sample k_t then multiply by GHI_clearsky to recover W/m².
#
# The score decoder now takes covariate-based conditioning and learns a
# cloud-modulation distribution, which is the physically meaningful quantity.
#
# Training:  k_raw in [0, 1.5] -> normalize to [-1, 1] via / KT_SCALE * 2 - 1
# Sampling:  sampled k_norm -> denormalize -> multiply by GHI_clearsky(t+h)

GHI_SCALE = 1200.0   # kept for legacy; only used if predict_kt=False
KT_SCALE = 1.5       # clear-sky index bound [0, 1.5]

class ScoreRes(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d), nn.SiLU(inplace=True), nn.Linear(d, d))
    def forward(self, x): return x + self.net(x)

class ScoreNet(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, h=256, blocks=2):
        super().__init__()
        d_in = 1 + 1 + z_dim + 1 + c_dim
        layers = [nn.Linear(d_in, h), nn.SiLU(inplace=True)]
        for _ in range(blocks): layers.append(ScoreRes(h))
        layers.append(nn.Linear(h, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, g, s, z, cti, c):
        return self.net(torch.cat([g, s, z, cti, c], dim=-1))

class CondScoreDecoder(nn.Module):
    """Score decoder that predicts clear-sky index k_t (not raw GHI).

    Pass predict_kt=True (default) to predict k_t; targets should be k_t values.
    The SolarSDE pipeline constructs k_t = ghi/ghi_clearsky at training time
    and reconstructs W/m² at sampling time via ghi_clearsky_target * k_t.
    """
    def __init__(self, z_dim=64, c_dim=5, h=256, blocks=2, steps=100, b0=1e-4, b1=0.02,
                 predict_kt=True):
        super().__init__()
        self.steps = steps
        self.predict_kt = predict_kt
        self.target_scale = KT_SCALE if predict_kt else GHI_SCALE
        self.score = ScoreNet(z_dim, c_dim, h, blocks)
        betas = torch.linspace(b0, b1, steps)
        alphas = 1 - betas
        ac = torch.cumprod(alphas, dim=0)
        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alphas_cum", ac)
        self.register_buffer("sac", torch.sqrt(ac))
        self.register_buffer("s1mac", torch.sqrt(1 - ac))

    def _normalize(self, y_raw):
        # y_raw in [0, scale] -> [-1, 1]
        return y_raw / self.target_scale * 2.0 - 1.0

    def _denormalize(self, y_norm):
        return (y_norm + 1.0) / 2.0 * self.target_scale

    def training_loss(self, target_raw, z, cti, c):
        """target_raw: k_t values (if predict_kt) or raw GHI (if not)."""
        t_norm = self._normalize(target_raw)
        B = t_norm.shape[0]
        si = torch.randint(0, self.steps, (B,), device=t_norm.device)
        sn = (si.float() / self.steps).unsqueeze(-1)
        eps = torch.randn_like(t_norm)
        ts = self.sac[si].unsqueeze(-1) * t_norm + self.s1mac[si].unsqueeze(-1) * eps
        pred_noise = self.score(ts, sn, z, cti, c)
        return {"loss": F.mse_loss(pred_noise, eps)}

    @torch.no_grad()
    def sample(self, z, cti, c, n=1):
        """Returns samples in the target scale.
           If predict_kt=True: returns k_t in [0, KT_SCALE], caller multiplies by ghi_clearsky
           Else: returns GHI in W/m² (legacy mode).
        """
        B = z.shape[0]
        z_e = z.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        cti_e = cti.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        c_e = c.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        x = torch.randn(B * n, 1, device=z.device)
        for i in reversed(range(self.steps)):
            sn = torch.full((B * n, 1), i / self.steps, device=z.device)
            eps_pred = self.score(x, sn, z_e, cti_e, c_e)
            b, a, ac = self.betas[i], self.alphas[i], self.alphas_cum[i]
            mean = (1 / a.sqrt()) * (x - b / (1 - ac).sqrt() * eps_pred)
            if i > 0:
                x = mean + b.sqrt() * torch.randn_like(x)
            else:
                x = mean
        y_out = self._denormalize(x).clamp(min=0.0, max=self.target_scale)
        return y_out.view(B, n)


## 6. Train Score Decoder

In [ ]:
torch.manual_seed(42)
score = CondScoreDecoder(z_dim=Z_DIM, c_dim=C_DIM, h=256, blocks=2, steps=100).to(DEVICE)
opt = torch.optim.Adam(score.parameters(), lr=2e-4)
# Cosine-annealing LR schedule: warm lr for first 10%, decay to 1e-5 over remaining epochs
print(f"Score Decoder params: {sum(p.numel() for p in score.parameters()):,}")

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False)
EPOCHS_SCORE = 150   # increased from 40 — DDPMs need more training than typical supervised models
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_SCORE, eta_min=1e-5)

best_val = float("inf"); t0 = time.time(); hist = []
for ep in range(1, EPOCHS_SCORE + 1):
    score.train(); tl = 0; n = 0
    for b in train_loader:
        z = b["z_t"].to(DEVICE); cti = b["cti"].to(DEVICE).unsqueeze(-1)
        c = b["cov"].to(DEVICE); g = b["ghi"].to(DEVICE).unsqueeze(-1)
        l = score.training_loss(g, z, cti, c)["loss"]
        opt.zero_grad(); l.backward()
        torch.nn.utils.clip_grad_norm_(score.parameters(), 1.0)
        opt.step()
        tl += l.item(); n += 1
    tl /= n
    sched.step()
    score.eval(); vl = vn = 0
    with torch.no_grad():
        for b in val_loader:
            z = b["z_t"].to(DEVICE); cti = b["cti"].to(DEVICE).unsqueeze(-1)
            c = b["cov"].to(DEVICE); g = b["ghi"].to(DEVICE).unsqueeze(-1)
            vl += score.training_loss(g, z, cti, c)["loss"].item(); vn += 1
    vl /= max(vn, 1)
    hist.append({"epoch": ep, "train_loss": tl, "val_loss": vl, "lr": opt.param_groups[0]["lr"]})
    if ep % 10 == 0 or ep == 1:
        print(f"Epoch {ep:3d}/{EPOCHS_SCORE} | train={tl:.4f} | val={vl:.4f} | lr={opt.param_groups[0]['lr']:.2e} | {(time.time()-t0)/60:.1f} min")
    if vl < best_val:
        best_val = vl; torch.save(score.state_dict(), CHECKPOINT_DIR / "score_best.pt")

torch.save(score.state_dict(), CHECKPOINT_DIR / "score_final.pt")
pd.DataFrame(hist).to_csv(RESULTS_DIR / "score_training_history.csv", index=False)
print(f"Score training complete. Best val: {best_val:.4f}. Time: {(time.time()-t0)/60:.1f} min")


## 7. Metrics + SDE solver

In [ ]:
# ==== Probabilistic forecasting metrics ====
def crps_empirical(y_true, y_samples):
    """CRPS from empirical samples. y_true: (N,), y_samples: (N, M)."""
    N, M = y_samples.shape
    t1 = np.mean(np.abs(y_samples - y_true[:, None]), axis=1)
    ys = np.sort(y_samples, axis=1)
    w = 2 * np.arange(1, M + 1) - M - 1
    t2 = np.sum(w[None, :] * ys, axis=1) / (M * M)
    return t1 - t2

def picp(y_true, y_samples, alpha=0.9):
    lo = np.quantile(y_samples, (1 - alpha) / 2, axis=1)
    hi = np.quantile(y_samples, 1 - (1 - alpha) / 2, axis=1)
    return float(((y_true >= lo) & (y_true <= hi)).mean())

def pinaw(y_samples, y_range, alpha=0.9):
    lo = np.quantile(y_samples, (1 - alpha) / 2, axis=1)
    hi = np.quantile(y_samples, 1 - (1 - alpha) / 2, axis=1)
    return float((hi - lo).mean() / max(y_range, 1e-9))

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def all_metrics(y_true, y_samples, is_ramp=None, alpha=0.9):
    y_med = np.median(y_samples, axis=1)
    y_range = y_true.max() - y_true.min()
    crps = crps_empirical(y_true, y_samples)
    out = {
        "crps": float(crps.mean()),
        "picp": picp(y_true, y_samples, alpha),
        "pinaw": pinaw(y_samples, y_range, alpha),
        "rmse": rmse(y_true, y_med),
        "mae":  mae(y_true, y_med),
    }
    if is_ramp is not None and is_ramp.sum() > 0:
        out["ramp_crps"] = float(crps[is_ramp].mean())
    return out


In [ ]:
# ==== Euler-Maruyama SDE solver with stability clamping ====
# The Neural SDE is trained on 1-step dynamics (SDE Matching); without clamping,
# long-horizon rollouts can drift out of the VAE latent distribution and the drift
# MLP will extrapolate catastrophically. We clamp per-step drift, diffusion, and
# the state vector to stay within ~4σ of training latent statistics.

# Compute training-latent stats once at module load.
# Use robust envelope: mean ± Z_CLAMP_STDS * std, with a per-dim minimum width so
# tiny-std latent dimensions don't get pinned to near-zero range at long horizons.
_train_Z_np = np.load(LATENT_DIR / "train_latents.npy")
Z_MEAN = torch.from_numpy(_train_Z_np.mean(0)).float().to(DEVICE)
Z_STD_RAW = torch.from_numpy(_train_Z_np.std(0)).float().to(DEVICE) + 1e-6
# Enforce minimum clamp width of 0.25 per dim so long-horizon rollouts aren't frozen
Z_STD = torch.maximum(Z_STD_RAW, torch.full_like(Z_STD_RAW, 0.05))
Z_CLAMP_STDS = 8.0   # loosened from 4.0 — prevents sample collapse to boundary at long horizons
MU_CAP = 10.0        # loosened from 5.0
SIGMA_CAP = 5.0      # loosened from 2.0

def em_step(drift_fn, diff_fn, z, t, c, cti, dt):
    mu = drift_fn(z, t, c).clamp(-MU_CAP, MU_CAP)
    sigma = diff_fn(z, cti).clamp(0.0, SIGMA_CAP)
    z_new = z + mu * dt + sigma * (dt ** 0.5) * torch.randn_like(z)
    return torch.clamp(z_new, Z_MEAN - Z_CLAMP_STDS * Z_STD, Z_MEAN + Z_CLAMP_STDS * Z_STD)

def solve_sde_horizons(sde, z0, horizons, c, cti, N=50, dt=1.0):
    B, d = z0.shape
    mx = max(horizons)
    hset = set(horizons)
    z = z0.unsqueeze(1).expand(B, N, d).reshape(B * N, d)
    c_e = c.unsqueeze(1).expand(B, N, -1).reshape(B * N, -1)
    cti_e = cti.unsqueeze(1).expand(B, N, -1).reshape(B * N, -1)
    out = {}
    for step in range(mx):
        t = torch.full((B * N, 1), float(step), device=z0.device)
        z = em_step(sde.drift, sde.diffusion, z, t, c_e, cti_e, dt)
        if (step + 1) in hset:
            out[step + 1] = z.view(B, N, d).clone()
    return out


## 8. Main evaluation at multiple horizons
Horizons: 6 steps (1 min), 30 steps (5 min), 60 steps (10 min), 120 steps (20 min), 180 steps (30 min). 50 MC samples each.

In [ ]:
# Load best checkpoints
sde.load_state_dict(torch.load(CHECKPOINT_DIR / "sde_best.pt", map_location=DEVICE))
score.load_state_dict(torch.load(CHECKPOINT_DIR / "score_best.pt", map_location=DEVICE))
sde.eval(); score.eval()

te = data["test"]
HORIZONS = [6, 30, 60, 120, 180]   # steps x 10s = 1, 5, 10, 20, 30 min
HORIZON_MIN = {6: 1, 30: 5, 60: 10, 120: 20, 180: 30}
N_SAMPLES = 50
N_EVAL = min(1000, len(te["Z"]) - max(HORIZONS) - 1)

print(f"Evaluating on {N_EVAL} test points with {N_SAMPLES} MC samples at horizons {list(HORIZON_MIN.values())} min")

results_by_horizon = {}
for h in HORIZONS:
    print(f"\nHorizon {HORIZON_MIN[h]} min ({h} steps) ...")
    y_true_list, y_samp_list, ramp_list = [], [], []
    batch = 32
    for i in tqdm(range(0, N_EVAL, batch), desc=f"h={HORIZON_MIN[h]}min"):
        idx = list(range(i, min(i + batch, N_EVAL)))
        z0 = torch.from_numpy(te["Z"][idx]).float().to(DEVICE)
        c = torch.from_numpy(te["cov"][idx]).float().to(DEVICE)
        cti = torch.from_numpy(te["cti"][idx]).float().unsqueeze(-1).to(DEVICE)
        with torch.no_grad():
            endp = solve_sde_horizons(sde, z0, [h], c, cti, N=N_SAMPLES, dt=1.0)[h]  # (B, N, d)
            B, N, d = endp.shape
            z_f = endp.view(B * N, d)
            cti_f = cti.unsqueeze(1).expand(B, N, -1).reshape(B * N, -1)
            c_f = c.unsqueeze(1).expand(B, N, -1).reshape(B * N, -1)
            g = score.sample(z_f, cti_f, c_f, n=1).squeeze(-1).view(B, N).cpu().numpy()
        for k, ii in enumerate(idx):
            target = ii + h
            if target < len(te["ghi"]):
                y_true_list.append(te["ghi"][target])
                y_samp_list.append(g[k])
                ramp_list.append(te["ramp"][target])
    yt = np.array(y_true_list); ys = np.array(y_samp_list); rm = np.array(ramp_list)
    m = all_metrics(yt, ys, is_ramp=rm)
    m["horizon_min"] = HORIZON_MIN[h]; m["horizon_steps"] = h
    m["n_eval"] = len(yt)
    m.setdefault("ramp_crps", 0.0)
    results_by_horizon[h] = m
    print(f"  CRPS={m['crps']:.3f}  RMSE={m['rmse']:.2f}  MAE={m['mae']:.2f}  "
          f"PICP={m['picp']:.3f}  PINAW={m['pinaw']:.3f}  Ramp-CRPS={m['ramp_crps']:.3f}")

df_res = pd.DataFrame.from_dict(results_by_horizon, orient="index").sort_values("horizon_min")
df_res.to_csv(RESULTS_DIR / "solar_sde_main_results.csv", index=False)
print("\nSolarSDE main results:")
cols_show = [c for c in ["horizon_min", "crps", "rmse", "mae", "picp", "pinaw", "ramp_crps"] if c in df_res.columns]
print(df_res[cols_show].to_string(index=False))


## 9. Zip outputs

In [ ]:
# ==== Zip outputs and prepare download ====
import shutil
zip_path = WORK_DIR / "solarsde_outputs.zip"
if zip_path.exists(): zip_path.unlink()
print(f"Zipping {PERSIST_DIR} -> {zip_path}")
shutil.make_archive(str(zip_path).replace(".zip", ""), "zip", root_dir=PERSIST_DIR)
size_mb = zip_path.stat().st_size / 1e6
print(f"Archive size: {size_mb:.1f} MB")

if IN_COLAB:
    from google.colab import files
    try:
        files.download(str(zip_path))
        print("Download triggered (check browser).")
    except Exception as e:
        print(f"Auto-download unavailable: {e}. Download manually from {zip_path}")
else:
    print(f"Archive at: {zip_path}  — download via Kaggle output tab or file browser.")


In [ ]:
print("=" * 70)
print("NOTEBOOK 2 COMPLETE")
print("=" * 70)
print("Trained: Neural SDE, Score Decoder")
print(f"Main results saved to: {RESULTS_DIR / 'solar_sde_main_results.csv'}")
print("Next: 03_baselines.ipynb to train comparison methods.")
